# Cuaderno de pruebas: frutas frescas y dañadas
**Desarrollado por:** Sebastián Alvarado, Daniel Pacheco y Cinthya Ramón

Funciones de visión por computador aplicadas a las fotos de `FOTOS`. Las figuras se guardan en `salidas/`.

In [ ]:
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

DATOS = Path('../data/estructurada')
SALIDAS = Path('salidas'); SALIDAS.mkdir(exist_ok=True)
etiquetas = pd.read_csv(DATOS / 'etiquetas.csv')

def elegir(clase, estado, n=1, semilla=0):
    filas = etiquetas[(etiquetas.clase == clase) & (etiquetas.estado == estado)]
    return [DATOS / a for a in filas.sample(n, random_state=semilla).archivo]

def leer(ruta, lado=256):
    img = cv2.imread(str(ruta))  # BGR
    return cv2.resize(img, (lado, lado))  # mismo tamaño para comparar

def rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

## 1. Elegir las fotos

In [ ]:
FOTOS = elegir('banano', 'fresca') + elegir('banano', 'dañada') + elegir('fresa', 'fresca') + elegir('fresa', 'dañada')
imgs = [leer(r) for r in FOTOS]
nombres = [r.stem.split('_')[0] + ' ' + r.stem.split('_')[1] for r in FOTOS]

def ver(lista, titulos, archivo, cmap=None):
    fig, ax = plt.subplots(1, len(lista), figsize=(3 * len(lista), 3), squeeze=False)
    for a, im, t in zip(ax[0], lista, titulos):
        a.imshow(im if im.ndim == 2 else rgb(im), cmap=cmap or 'gray'); a.set_title(t); a.axis('off')
    plt.tight_layout(); plt.savefig(SALIDAS / f'{archivo}.png', dpi=150); plt.show()

ver(imgs, nombres, 'fotos')

## 2. La imagen como matriz

In [ ]:
def info(ruta):
    img = cv2.imread(str(ruta))
    alto, ancho, canales = img.shape
    print(f'{ruta.name}: {ancho}x{alto}, {canales} canales, {img.dtype} -> {8 * canales} bits por píxel, '
          f'{img.nbytes / 1e3:.0f} KB en memoria y {ruta.stat().st_size / 1e3:.0f} KB en disco')
    print('píxel del centro (B, G, R):', img[alto // 2, ancho // 2])

for r in FOTOS: info(r)

## 3. Espacios de color

HSV separa matiz (H), saturación (S) y brillo (V); RGB mezcla color y luz.

In [ ]:
def espacios(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    b, g, r = cv2.split(img); h, s, v = cv2.split(hsv)
    ver([img, cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), r, g, b, h, s, v],
        ['original', 'gris', 'R', 'G', 'B', 'H', 'S', 'V'], 'espacios')

espacios(imgs[1])

### Histogramas comparados

Una curva por foto: el canal donde fresca y dañada se separan es el que distingue el estado.

In [ ]:
def histogramas(imgs, nombres):
    canales = {'gris': lambda im: cv2.cvtColor(im, cv2.COLOR_BGR2GRAY),
               'H': lambda im: cv2.cvtColor(im, cv2.COLOR_BGR2HSV)[..., 0],
               'S': lambda im: cv2.cvtColor(im, cv2.COLOR_BGR2HSV)[..., 1],
               'V': lambda im: cv2.cvtColor(im, cv2.COLOR_BGR2HSV)[..., 2],
               'R': lambda im: im[..., 2], 'G': lambda im: im[..., 1], 'B': lambda im: im[..., 0]}
    fig, ax = plt.subplots(1, len(canales), figsize=(22, 3))
    for a, (nombre, f) in zip(ax, canales.items()):
        tope = 180 if nombre == 'H' else 256  # H va de 0 a 179 en OpenCV
        for im, n in zip(imgs, nombres):
            a.plot(cv2.calcHist([f(im)], [0], None, [32], [0, tope]).ravel() / im[..., 0].size, label=n)
        a.set_title(nombre)
    ax[0].legend(fontsize=7); plt.tight_layout(); plt.savefig(SALIDAS / 'histogramas.png', dpi=150); plt.show()

histogramas(imgs, nombres)

## 4. Operaciones de punto

Cada píxel nuevo depende solo de su valor: normalización (estira a 0–255), α·I + β (contraste y brillo), gamma (I^γ) y negativo (255 − I).

In [ ]:
def punto(img, alfa=1.4, beta=-30, g=0.5):
    gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    salidas = {'gris': gris, 'normalizada': cv2.normalize(gris, None, 0, 255, cv2.NORM_MINMAX),
               f'α={alfa} β={beta}': cv2.convertScaleAbs(gris, alpha=alfa, beta=beta),
               f'γ={g}': (255 * (gris / 255) ** g).astype(np.uint8), 'negativo': 255 - gris}
    fig, ax = plt.subplots(2, len(salidas), figsize=(3 * len(salidas), 5))
    for i, (n, s) in enumerate(salidas.items()):
        ax[0, i].imshow(s, cmap='gray', vmin=0, vmax=255); ax[0, i].set_title(n); ax[0, i].axis('off')
        ax[1, i].hist(s.ravel(), 64, (0, 256), color='gray')
    plt.tight_layout(); plt.savefig(SALIDAS / 'punto.png', dpi=150); plt.show()

punto(imgs[1])

## 5. Preprocesamiento

### Ecualización global contra adaptativa (CLAHE)

Solo sobre V, para no cambiar el color. La global usa todo el histograma; CLAHE trabaja por zonas.

In [ ]:
def ecualizar(img):
    h, s, v = cv2.split(cv2.cvtColor(img, cv2.COLOR_BGR2HSV))
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    a_bgr = lambda v2: cv2.cvtColor(cv2.merge([h, s, v2]), cv2.COLOR_HSV2BGR)
    ver([img, a_bgr(cv2.equalizeHist(v)), a_bgr(clahe.apply(v))], ['original', 'global', 'CLAHE'], 'ecualizacion')

ecualizar(imgs[0])

### Ruido y filtros

PSNR (dB): parecido con la original, más alto es mejor.

In [ ]:
def filtros(img, ruido='sal_pimienta', k=5):
    rng = np.random.default_rng(0)
    if ruido == 'sal_pimienta':
        sucia = img.copy(); p = rng.random(img.shape[:2])
        sucia[p < 0.03] = 0; sucia[p > 0.97] = 255
    else:
        sucia = np.clip(img + rng.normal(0, 20, img.shape), 0, 255).astype(np.uint8)
    salidas = {'con ruido': sucia, 'media': cv2.blur(sucia, (k, k)),
               'mediana': cv2.medianBlur(sucia, k), 'gaussiano': cv2.GaussianBlur(sucia, (k, k), 0)}
    ver(list(salidas.values()), [f'{n}\nPSNR {cv2.PSNR(img, s):.1f}' for n, s in salidas.items()], f'filtros_{ruido}')

filtros(imgs[1], 'sal_pimienta')
filtros(imgs[1], 'gaussiano')

## 6. Segmentación clásica

Otsu en S separa la fruta del fondo claro; apertura y cierre limpian la máscara; el contorno más grande da el recorte.

In [ ]:
def segmentar(img, mostrar=None):
    s = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)[..., 1]
    _, mascara = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    limpia = cv2.morphologyEx(cv2.morphologyEx(mascara, cv2.MORPH_OPEN, k), cv2.MORPH_CLOSE, k)
    bordes = cv2.Canny(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), 50, 150)
    contornos, _ = cv2.findContours(limpia, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    x, y, w, h = cv2.boundingRect(max(contornos, key=cv2.contourArea))
    if mostrar:  # mostrar = nombre del archivo de salida
        caja = img.copy(); cv2.rectangle(caja, (x, y), (x + w, y + h), (0, 255, 0), 2)
        ver([img, mascara, limpia, bordes, caja, img[y:y + h, x:x + w]],
            ['original', 'Otsu en S', 'morfología', 'Canny', 'contorno', 'recorte'], f'segmentacion_{mostrar}')
    return limpia

for im, n in zip(imgs, nombres): segmentar(im, n.replace(' ', '_'))

## 7. Características

Color medio (HSV), forma (Hu), textura (LBP) y gradientes (HOG).

In [ ]:
def lbp(gris):
    """Compara cada píxel con sus 8 vecinos: 1 si el vecino es mayor o igual. Los 8 bits forman un código de textura."""
    c = gris[1:-1, 1:-1].astype(int); codigo = np.zeros_like(c)
    vecinos = [(-1, -1), (-1, 0), (-1, 1), (0, 1), (1, 1), (1, 0), (1, -1), (0, -1)]
    for bit, (dy, dx) in enumerate(vecinos):
        v = gris[1 + dy:gris.shape[0] - 1 + dy, 1 + dx:gris.shape[1] - 1 + dx]
        codigo += (v >= c) << bit
    return codigo.astype(np.uint8)

def caracteristicas(img, nombre):
    gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY); mascara = segmentar(img)
    hsv_medio = cv2.mean(cv2.cvtColor(img, cv2.COLOR_BGR2HSV), mascara)[:3]
    hu = cv2.HuMoments(cv2.moments(mascara)).ravel()
    hog = cv2.HOGDescriptor((64, 64), (16, 16), (8, 8), (8, 8), 9).compute(cv2.resize(gris, (64, 64)))
    gx, gy = cv2.Sobel(gris, cv2.CV_32F, 1, 0), cv2.Sobel(gris, cv2.CV_32F, 0, 1)
    print(f'HSV medio: {np.round(hsv_medio, 1)}   Hu[0:3]: {np.round(-np.sign(hu[:3]) * np.log10(np.abs(hu[:3])), 2)}   HOG: {hog.size} valores')
    ver([img, lbp(gris), cv2.magnitude(gx, gy)], ['original', 'LBP (textura)', 'gradiente (base de HOG)'], f'caracteristicas_{nombre}')

for im, n in zip(imgs, nombres): caracteristicas(im, n.replace(' ', '_'))